pip install sentence-transformers faiss-cpu pypdf #imported

pip install python-docx

Extract resume text

In [2]:
from pypdf import PdfReader

reader = PdfReader("me/resume.pdf")

full_text = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        full_text += text + "\n"

print(full_text[:1000])

 
SIVA SANKARI SIVAKAMINATHAN 
 
Siva Sankari Sivakaminathan 
Flat 26, Shaftesbury apartments, 28 Mount Pleasant, Liverpool L3 5SA 
 +44 7393068337 | sankari.s2009@gmail.com | https://www.linkedin.com/in/siva-sankari-sivakaminathan/ 
PROFILE 
Experienced and results-driven Data Scientist with a Master’s in Data Science and Communication, currently leading 
an AI Knowledge Transfer Partnership at Harper James. Specializing in generative AI, semantic content analytics, 
multimodal data processing, and neural networks, with a strong focus on designing scalable, privacy-preserving 
solutions for the legal sector. Skilled in managing end-to-end AI initiatives from requirements gathering and 
stakeholder engagement to model development, evaluation, and deployment while ensuring alignment with business 
objectives. Proven ability to bridge the gap between technical innovation and operational needs, enabling cross-
functional teams to adopt AI-driven tools for improved efficiency, compliance, 

Clean Text

In [3]:
import re

clean_text = re.sub(r'\n+', '\n', full_text)
clean_text = re.sub(r'\s+', ' ', clean_text)

print(clean_text[:1000])

 SIVA SANKARI SIVAKAMINATHAN Siva Sankari Sivakaminathan Flat 26, Shaftesbury apartments, 28 Mount Pleasant, Liverpool L3 5SA +44 7393068337 | sankari.s2009@gmail.com | https://www.linkedin.com/in/siva-sankari-sivakaminathan/ PROFILE Experienced and results-driven Data Scientist with a Master’s in Data Science and Communication, currently leading an AI Knowledge Transfer Partnership at Harper James. Specializing in generative AI, semantic content analytics, multimodal data processing, and neural networks, with a strong focus on designing scalable, privacy-preserving solutions for the legal sector. Skilled in managing end-to-end AI initiatives from requirements gathering and stakeholder engagement to model development, evaluation, and deployment while ensuring alignment with business objectives. Proven ability to bridge the gap between technical innovation and operational needs, enabling cross- functional teams to adopt AI-driven tools for improved efficiency, compliance, and decision-m

Chunking

In [4]:
def chunk_text(text, chunk_size=400, overlap=100):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks

chunks = chunk_text(clean_text)

len(chunks)

25

In [5]:
chunks[:2]

[' SIVA SANKARI SIVAKAMINATHAN Siva Sankari Sivakaminathan Flat 26, Shaftesbury apartments, 28 Mount Pleasant, Liverpool L3 5SA +44 7393068337 | sankari.s2009@gmail.com | https://www.linkedin.com/in/siva-sankari-sivakaminathan/ PROFILE Experienced and results-driven Data Scientist with a Master’s in Data Science and Communication, currently leading an AI Knowledge Transfer Partnership at Harper Jame',
 'ata Science and Communication, currently leading an AI Knowledge Transfer Partnership at Harper James. Specializing in generative AI, semantic content analytics, multimodal data processing, and neural networks, with a strong focus on designing scalable, privacy-preserving solutions for the legal sector. Skilled in managing end-to-end AI initiatives from requirements gathering and stakeholder engag']

Create Embeddings

In [6]:
from openai import OpenAI
import numpy as np
import os
from dotenv import load_dotenv

load_dotenv()

client = OpenAI()

def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

In [7]:
embeddings = [get_embedding(chunk) for chunk in chunks]

len(embeddings)

25

Convert to numpy metrics

In [8]:
embedding_matrix = np.array(embeddings)
embedding_matrix.shape

(25, 1536)

Test Retreival with cosine similarity

In [9]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve(query, k=3):
    query_embedding = np.array(get_embedding(query)).reshape(1, -1)

    similarities = cosine_similarity(query_embedding, embedding_matrix)[0]

    top_indices = similarities.argsort()[-k:][::-1]

    return [chunks[i] for i in top_indices]

retrieve("What machine learning experience does she have?")

['enerative AI: Experience in building text generation/ summarization models and anomaly detection models using GANs, VAEs and autoencoders. ● Natural Language Processing (NLP) & Large Language Models (LLMs): Strong foundation in NLP techniques, including tokenization, stemming, lemmatization, and part-of-speech tagging, with hands-on experience using NLTK and SpaCy. ● Data Analysis & Visualization:',
 'ns. ● Gained hands-on experience with big data analysis, exploratory data analysis (EDA), and visualization, contributing to projects in areas like customer segmentation and movie preferences analysis. WORK EXPERIENCE AI Engineer – KTP Associate | Aston University & Harper James Ltd Apr 2025 till date \uf0b7 Leading the design and development of an AI-powered semantic content analytics platform to enha',
 'cross- functional teams to adopt AI-driven tools for improved efficiency, compliance, and decision-making. TECHNICAL SKILLS ● Artificial Intelligence & Machine Learning: Proficient in

In [10]:
np.save("resume_chunks.npy", np.array(chunks, dtype=object))
np.save("resume_embeddings.npy", embedding_matrix)

print("Saved resume_chunks.npy and resume_embeddings.npy")

Saved resume_chunks.npy and resume_embeddings.npy
